# GRPO Fine-Tuning on the AWS RL Environment

Continues training the [`Sizzing/aws-rl-sft-qwen25coder3b-adapter`](https://huggingface.co/Sizzing/aws-rl-sft-qwen25coder3b-adapter) with **Group Relative Policy Optimization** where the reward comes from a **live AWS RL environment** (hosted elsewhere, reached via HTTPS). Hyperparameter-tuned with Optuna, logged to Weights & Biases, checkpointed for safe resume, and published to a **separate** HuggingFace Hub repo so both the SFT and GRPO adapters coexist for side-by-side comparison.

### Architecture
```
Hosted env server (Docker, AWS_RL_ENV_POOL_SIZE=8)
        ▲ HTTPS via cloudflared / ngrok tunnel
        │                                 Colab / Kaggle VM (T4, 15.6 GB VRAM)
        │                                  └─ main python
        │  8-way httpx pool (reward_fn)         ├─ Unsloth: base Qwen2.5-Coder-3B (4-bit)
        ├──────────────────────────       │   + SFT adapter (trainable)
        │                                       ├─ TRL GRPOTrainer
        │  8-way GrpoPool WS (multi-step eval)  │    ├─ G generations per prompt
        └──────────────────────────       │    ├─ reward_fn → 8 concurrent /reset + /step
                                                │    └─ group-advantage + PPO-clip on LoRA
                                                ├─ Optuna (sqlite, resumable): {lr, β, G, T, r, α_mul}
                                                └─ push → Sizzing/aws-rl-grpo-qwen25coder3b-adapter
```

### Why GRPO on top of SFT?
SFT teaches the model **what a good AWS CLI command looks like**. GRPO teaches the model **which commands actually work against the environment** — the reward is grounded in real (simulated) AWS state transitions, not string matching. We keep training single-step for TRL-native integration, then measure with full multi-step rollouts (chaos, drift, hints, recovery).

### Headline metrics (reported in the eval cells)
| Axis                 | SFT baseline | GRPO target | Stretch |
|----------------------|-------------:|------------:|--------:|
| `env_reward_mean`    |        ~0.85 |        ≥0.92 |   ≥0.97 |
| `env_success_rate`   |         ~85% |         ≥92% |    ≥97% |
| `recovery_rate`      |         ~40% |         ≥60% |    ≥75% |
| `drift_repair_rate`  |         ~30% |         ≥55% |    ≥70% |
| `hints_per_solved`   |         ~0.8 |        ≤0.5 |    ≤0.2 |
| `steps_to_solve`     |           ~6 |          ≤5 |      ≤4 |

## 1 · Install dependencies

Mirrors the SFT notebook's pinning strategy. Key constraints: `trl>=0.16` for `GRPOTrainer`, `transformers>=4.50,<5.0` for Unsloth compatibility (the SFT notebook ran into this too), `--force-reinstall --no-deps` on `transformers` so TRL doesn't pull an incompatible version.

In [ ]:
%%capture
%pip install -q --upgrade pip
%pip install -q "unsloth"
%pip install -q "trl>=0.16,<0.18" "peft" "accelerate" "datasets" "bitsandbytes"
%pip install -q "transformers>=4.50,<5.0" --force-reinstall --no-deps
%pip install -q "optuna" "wandb" "plotly" "kaleido"
%pip install -q "httpx" "websockets" "pyyaml" "python-dotenv"
%pip install -q "openenv-core" "huggingface_hub>=1.9.0"

## 2 · Clone the AWS RL env repo

We need the `client.AwsRlEnv` class, `models.py` pydantic types, `scripts/grpo_pool.py`, the curriculum loader, and the task YAMLs — these power the multi-step evaluator. We do **not** install or run any env server locally; the env is hosted elsewhere.

In [ ]:
from __future__ import annotations
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/UdayKiranPadhy/aws-rl-env.git"
REPO_DIR = Path("/content/aws-rl-env" if Path("/content").exists() else "/kaggle/working/aws-rl-env")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Repo at {REPO_DIR} — ready.")

## 3 · Runtime detection

Matches the SFT notebook so we pick `fp16` on T4 (bf16 unsupported) and `bf16` on A100/H100. Also sets the PyTorch allocator env var that cut OOMs on long Unsloth runs.

In [ ]:
from dataclasses import dataclass
import torch

IS_COLAB  = Path("/content").exists() and not Path("/kaggle").exists()
IS_KAGGLE = Path("/kaggle").exists()

OUT_DIR = Path("/content/out") if IS_COLAB else Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")


@dataclass(frozen=True)
class Runtime:
    gpu_name: str
    use_fp16: bool
    use_bf16: bool


def detect_runtime() -> Runtime:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. This notebook needs CUDA (T4/A100/H100).")
    name = torch.cuda.get_device_name(0)
    is_t4 = "T4" in name
    return Runtime(gpu_name=name, use_fp16=is_t4, use_bf16=not is_t4)


RT = detect_runtime()
print(f"GPU: {RT.gpu_name}  |  fp16={RT.use_fp16}  bf16={RT.use_bf16}  |  OUT_DIR={OUT_DIR}")

## 4 · Training configuration

One frozen `TrainingConfig` dataclass holds every tunable hyperparameter. Optuna hands trials a mutated copy of this same dataclass, so the trial path and the final-run path go through identical code.

Separate `ModelSpec` and `PathsSpec` keep non-tunable identifiers out of the search space.

In [ ]:
from dataclasses import replace


@dataclass(frozen=True)
class ModelSpec:
    base_model:   str = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
    sft_adapter:  str = "Sizzing/aws-rl-sft-qwen25coder3b-adapter"
    grpo_adapter: str = "Sizzing/aws-rl-grpo-qwen25coder3b-adapter"   # NEW repo — SFT repo untouched
    dataset_repo: str = "Sizzing/aws-rl-sft"
    max_seq_length: int = 512


@dataclass(frozen=True)
class TrainingConfig:
    # GRPO knobs (Optuna-tunable)
    learning_rate:   float = 5e-6
    beta:            float = 0.04    # KL coef vs. reference (frozen SFT adapter)
    num_generations: int   = 8       # G in GRPO
    temperature:     float = 0.9
    top_p:           float = 0.95
    lora_r:          int   = 16
    lora_alpha_mul:  int   = 2
    # Empirical defaults — hold fixed
    max_prompt_length:     int = 384
    max_completion_length: int = 128
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs:      int = 1
    save_steps:            int = 25
    save_total_limit:      int = 3
    eval_steps:            int = 50
    warmup_ratio:          float = 0.05
    seed:                  int = 42


@dataclass(frozen=True)
class PipelineConfig:
    env_pool_size: int = 8
    n_trials:      int = 6
    trial_max_steps:       int = 30
    trial_train_subset:    int = 80
    val_subset_size:       int = 20
    eval_reserve_cap:      int = 100  # cap multi-step eval to this many tasks for speed
    wandb_project: str = "AWS-RL-GRPO"
    wandb_entity:  str = "sizzing-sizzing"


MODEL = ModelSpec()
TRAIN = TrainingConfig()
PIPE  = PipelineConfig()

print("ModelSpec:",   MODEL)
print("TrainingConfig defaults:", TRAIN)
print("PipelineConfig:", PIPE)

## 5 · Authenticate: HF Hub, Weights & Biases, env URL

Three secrets are required:

| Secret                  | Scope         | Used for                                 |
|-------------------------|---------------|------------------------------------------|
| `HF_TOKEN`              | read + write  | pull SFT adapter, push GRPO adapter      |
| `WANDB_API_KEY`         | default       | project logging                          |
| `AWS_RL_ENV_BASE_URL`   | —              | URL of the hosted env (cloudflared/ngrok) |

We also dummy-push a tiny file to verify the HF token has write access to the `Sizzing` org **before** spending hours training.

In [ ]:
def _load_secret(name: str) -> str:
    """Read a secret from Colab userdata, Kaggle UserSecrets, or the env."""
    if IS_COLAB:
        from google.colab import userdata
        try: return userdata.get(name)
        except Exception: pass
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try: return UserSecretsClient().get_secret(name)
        except Exception: pass
    val = os.environ.get(name)
    if not val:
        raise RuntimeError(f"Secret {name!r} is missing. Set it in Colab/Kaggle secrets or as an env var.")
    return val


HF_TOKEN         = _load_secret("HF_TOKEN")
WANDB_API_KEY    = _load_secret("WANDB_API_KEY")
ENV_BASE_URL     = _load_secret("AWS_RL_ENV_BASE_URL").rstrip("/")

os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
# Keep the tunnel URL out of wandb.config to avoid leaking it on the public dashboard
os.environ["WANDB_DISABLE_GIT"] = "true"

from huggingface_hub import login as hf_login, HfApi
import wandb

hf_login(token=HF_TOKEN, add_to_git_credential=False)
wandb.login(key=WANDB_API_KEY)


def verify_hub_write_scope(repo_id: str) -> None:
    """Ensure the HF token can create repos under the target org before training.

    Without this, we'd discover permission failures *after* a multi-hour run.
    """
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=repo_id, private=True, exist_ok=True, repo_type="model")
    probe = OUT_DIR / ".hub_write_probe.txt"
    probe.write_text("ok")
    api.upload_file(path_or_fileobj=str(probe), path_in_repo=".hub_write_probe.txt",
                    repo_id=repo_id, commit_message="probe: verify write scope")
    probe.unlink()


verify_hub_write_scope(MODEL.grpo_adapter)
print("All secrets loaded; HF write access to", MODEL.grpo_adapter, "verified.")

## 6 · Connect to the hosted env + health check

The env server runs **outside** this VM. We only assert reachability and that its internal pool has 8 slots (required for parallel rollouts). If either fails, we stop **before** loading the model.

In [ ]:
import httpx


def probe_env(base_url: str, required_pool: int) -> dict:
    """One-shot reachability + pool-size check. Raises on failure."""
    with httpx.Client(base_url=base_url, timeout=10.0) as c:
        schema = c.get("/schema").raise_for_status().json()
        state  = c.get("/state").raise_for_status().json()
    pool = int(state.get("pool_size") or state.get("pool") or 1)
    if pool < required_pool:
        raise RuntimeError(
            f"Hosted env has pool_size={pool} but we need {required_pool}. "
            "Restart the server with AWS_RL_ENV_POOL_SIZE=8."
        )
    return {"schema": schema, "pool_size": pool}


probe = probe_env(ENV_BASE_URL, required_pool=PIPE.env_pool_size)
print(f"Env online. Pool size: {probe['pool_size']}. Action schema keys: {list(probe['schema'].keys())}")

In [ ]:
# Smoke test: reset to a warmup task (task_id=0 is typically "list S3 buckets") and
# run `aws s3 ls`. Expect a reward near 1.0. This validates the full round trip
# before we spend GPU time on anything.
from models import Task
from server.services.curriculum import load_tier
from models import TaskDifficulty

_tasks_dir = REPO_DIR / "server" / "services" / "tasks"
_warmup_tasks = load_tier(TaskDifficulty.WARMUP, _tasks_dir)
_smoke_task = next(t for t in _warmup_tasks if "s3" in t.description.lower()
                   and "list" in t.description.lower())

with httpx.Client(base_url=ENV_BASE_URL, timeout=30.0) as c:
    c.post("/reset", json={"task": _smoke_task.model_dump()}).raise_for_status()
    r = c.post("/step", json={"action": {"command": "aws s3 ls"}}).raise_for_status().json()

print(f"Smoke task: {_smoke_task.description}\nReward: {r.get('reward'):.3f}  success={r.get('observation', {}).get('command_success')}")
assert r.get("reward", 0.0) > 0.5, "Smoke test reward too low — env may be misconfigured"
print("Env smoke test passed.")

## 7 · Dataset + task lookup map

GRPO needs prompts (not prompt-response pairs). We filter the SFT dataset to `step_idx == 0` rows — i.e. *first-step* prompts where the agent faces a fresh task with no prior history. Each generated completion will be scored by resetting the env to that task and executing the one command.

`TASK_MAP: dict[int, Task]` lets the reward function look up the full pydantic `Task` (needed to serialise over HTTP to `/reset`) by `task_id` alone.

In [ ]:
from datasets import load_dataset, Dataset
import yaml


def build_task_map(tasks_dir: Path) -> dict[int, Task]:
    """Load every task YAML into a dict keyed by task_id.

    The reward function only has task_id to work with; this map lets it
    recover the full Task (success_criteria, setup_commands, ...) needed
    for env /reset.
    """
    m: dict[int, Task] = {}
    for tier in TaskDifficulty:
        try:
            for t in load_tier(tier, tasks_dir):
                m[int(t.task_id)] = t
        except FileNotFoundError:
            continue
    return m


def load_drift_task_ids(tasks_dir: Path) -> set[int]:
    """Drift tasks live in drift.yaml and get merged into the EXPERT tier by
    the curriculum loader. We scan the file directly so we can still identify
    them in multi-step eval (drift_repair_rate). Falls back to detecting tasks
    with `desired_state_spec` set if drift.yaml is missing.
    """
    ids: set[int] = set()
    drift_path = tasks_dir / "drift.yaml"
    if drift_path.exists():
        with open(drift_path) as f:
            for entry in (yaml.safe_load(f) or []):
                ids.add(int(entry["task_id"]))
    return ids


def build_grpo_datasets(dataset_repo: str, task_map: dict[int, Task],
                        val_size: int, seed: int = 42):
    """Return (train_ds, val_ds, reserve_ds) with columns: prompt, task_id.

    `prompt` is the chat-style message list (system + user) so the tokenizer's
    chat template applies automatically inside GRPOTrainer. We drop the
    assistant row — GRPO generates its own.
    """
    raw = load_dataset(dataset_repo, token=HF_TOKEN)

    def _to_row(r):
        return {"prompt": r["messages"][:2], "task_id": int(r["task_id"])}

    def _single_turn(split):
        return raw[split].filter(lambda r: r["step_idx"] == 0
                                   and int(r["task_id"]) in task_map)

    train_single = _single_turn("train")
    train_ds = train_single.map(
        _to_row,
        remove_columns=[c for c in train_single.column_names if c not in ("prompt", "task_id")],
    )

    val_single = _single_turn("validation")
    val_ds = val_single.shuffle(seed=seed).select(range(min(val_size, len(val_single))))
    val_ds = val_ds.map(
        _to_row,
        remove_columns=[c for c in val_ds.column_names if c not in ("prompt", "task_id")],
    )

    reserve_ds = None
    if "reserve" in raw:
        reserve_single = _single_turn("reserve")
        reserve_ds = reserve_single.map(
            _to_row,
            remove_columns=[c for c in reserve_single.column_names if c not in ("prompt", "task_id")],
        )

    return train_ds, val_ds, reserve_ds


_tasks_dir = REPO_DIR / "server" / "services" / "tasks"
TASK_MAP = build_task_map(_tasks_dir)
DRIFT_TASK_IDS = load_drift_task_ids(_tasks_dir)
TRAIN_DS, VAL_DS, RESERVE_DS = build_grpo_datasets(
    MODEL.dataset_repo, TASK_MAP, val_size=PIPE.val_subset_size, seed=TRAIN.seed,
)

print(f"TASK_MAP:  {len(TASK_MAP)} tasks across {len({t.difficulty for t in TASK_MAP.values()})} tiers")
print(f"DRIFT_TASK_IDS: {len(DRIFT_TASK_IDS)} drift tasks")
print(f"TRAIN_DS:  {len(TRAIN_DS)}  VAL_DS: {len(VAL_DS)}  RESERVE_DS: {len(RESERVE_DS) if RESERVE_DS else 0}")

# Sanity: every task_id referenced in the training set must resolve
_missing = {r["task_id"] for r in TRAIN_DS if r["task_id"] not in TASK_MAP}
assert not _missing, f"Task IDs referenced in dataset but missing from TASK_MAP: {_missing}"

## 8 · Reward functions

Three reward functions passed to `GRPOTrainer.reward_funcs`:

| Reward           | Weight | Signal                                                     |
|------------------|-------:|------------------------------------------------------------|
| `env_reward`     |   1.0  | Real env reward from `/reset` + `/step` against the task   |
| `format_reward`  |  0.15  | 1.0 if completion starts with `aws `, else 0.0             |
| `length_reward`  |  0.05  | Soft length prior: 1.0 ≤120 chars, decays to 0.0 by 400    |

`EnvRewardClient` wraps all HTTP concerns (thread-local httpx, counters for wandb health logging). `build_reward_funcs` is a closure factory — it binds the client + `TASK_MAP` into three stateless callables so GRPOTrainer gets plain functions without any hidden module-level globals.

In [ ]:
import re, threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import field
from typing import Callable, Iterable


def extract_aws_command(raw: str) -> str:
    """Pure: first `aws ...` line from model output; fall back to `aws help`.

    Pure (no I/O) so it can be unit-tested and reused by the multi-step
    evaluator. `aws help` is a safe fallback: it always succeeds in the
    env, earning zero task reward but never crashing a batch.
    """
    for line in raw.splitlines():
        line = line.strip().strip("`").strip()
        if line.startswith("aws "):
            return line
    return "aws help"


@dataclass
class EnvRewardClient:
    """Thread-safe HTTP wrapper for the hosted env, with health counters."""
    base_url: str
    pool_size: int = 8
    timeout_s: float = 30.0
    _tls: threading.local = field(default_factory=threading.local, repr=False)
    _executor: ThreadPoolExecutor = field(init=False, repr=False)
    success: int = 0
    timeout: int = 0
    conn_err: int = 0

    def __post_init__(self) -> None:
        self._executor = ThreadPoolExecutor(max_workers=self.pool_size)

    def _client(self) -> httpx.Client:
        c = getattr(self._tls, "client", None)
        if c is None:
            c = httpx.Client(
                base_url=self.base_url,
                timeout=httpx.Timeout(self.timeout_s, connect=10.0),
            )
            self._tls.client = c
        return c

    def health(self) -> dict:
        return self._client().get("/schema").raise_for_status().json()

    def score_one(self, task: Task, command: str) -> float:
        try:
            c = self._client()
            c.post("/reset", json={"task": task.model_dump()}).raise_for_status()
            r = c.post("/step", json={"action": {"command": command}})
            r.raise_for_status()
            self.success += 1
            return float(r.json().get("reward", 0.0))
        except httpx.TimeoutException:
            self.timeout += 1
            return 0.0
        except httpx.HTTPError:
            self.conn_err += 1
            return 0.0

    def score_batch(self, tasks: list[Task], commands: list[str]) -> list[float]:
        """Parallel scoring; preserves input order."""
        return list(self._executor.map(self.score_one, tasks, commands))

    def drain_counters(self) -> dict[str, int]:
        """Return counters and reset. Used by the wandb callback every N steps.\""""
        out = {"success": self.success, "timeout": self.timeout, "conn_err": self.conn_err}
        self.success = self.timeout = self.conn_err = 0
        return out


def build_reward_funcs(env: EnvRewardClient,
                       task_map: dict[int, Task]) -> tuple[Callable, Callable, Callable]:
    """Return (env_reward, format_reward, length_reward) bound to `env`+`task_map`.

    TRL passes non-`prompt` dataset columns (here: `task_id`) as kwargs aligned
    with the completion list. We rely on that contract; `remove_unused_columns=False`
    in GRPOConfig is what preserves `task_id` end-to-end.
    """

    def _text_of(c) -> str:
        return c if isinstance(c, str) else c[0]["content"]

    def env_reward(prompts, completions, task_id=None, **kw):
        tids = task_id if task_id is not None else kw.get("task_id", [])
        tasks = [task_map[int(t)] for t in tids]
        cmds  = [extract_aws_command(_text_of(c)) for c in completions]
        return env.score_batch(tasks, cmds)

    def format_reward(prompts, completions, **kw):
        return [1.0 if _text_of(c).lstrip().startswith("aws ") else 0.0 for c in completions]

    def length_reward(prompts, completions, **kw):
        out = []
        for c in completions:
            n = len(_text_of(c))
            out.append(1.0 if n <= 120 else max(0.0, 1.0 - (n - 120) / 280.0))
        return out

    return env_reward, format_reward, length_reward


ENV_CLIENT = EnvRewardClient(base_url=ENV_BASE_URL, pool_size=PIPE.env_pool_size)
ENV_REWARD, FORMAT_REWARD, LENGTH_REWARD = build_reward_funcs(ENV_CLIENT, TASK_MAP)

# Quick smoke test: score `aws s3 ls` on the warmup S3-list task. Expect ~1.0.
_smoke_rewards = ENV_REWARD(
    prompts=[None] * 2,
    completions=["aws s3 ls", "aws s3 ls"],
    task_id=[_smoke_task.task_id, _smoke_task.task_id],
)
print(f"Reward smoke test on task {_smoke_task.task_id}: {_smoke_rewards}")
assert min(_smoke_rewards) > 0.5, "Reward smoke test failed — env or reward wiring broken"

## 9 · Load the SFT adapter as the starting policy

We go through `PeftModel.from_pretrained(base, sft_adapter, is_trainable=True)` explicitly (rather than `FastLanguageModel.from_pretrained(adapter_repo)`) so there is no ambiguity about the adapter landing in trainable mode — GRPO must be able to update the LoRA weights.

This helper is also used later by the final-run and Optuna-trial paths, so it lives in its own cell.

In [ ]:
import gc


def load_policy(spec: ModelSpec, trainable: bool = True):
    """Load base (4-bit) + SFT adapter.

    `trainable=True` gives a PeftModel with is_trainable=True — the correct
    mode for GRPO. `trainable=False` loads the adapter for inference only
    and invokes `FastLanguageModel.for_inference` for Unsloth's fused
    inference kernels (used during evaluation).
    """
    from unsloth import FastLanguageModel
    from peft import PeftModel

    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=spec.base_model,
        max_seq_length=spec.max_seq_length,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(base, spec.sft_adapter, is_trainable=trainable)
    if trainable:
        FastLanguageModel.for_training(model)
    else:
        FastLanguageModel.for_inference(model)
    return model, tokenizer


def free_model(model) -> None:
    """Release GPU memory held by a model + its optimizer state."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


# Sanity: load + confirm LoRA params are trainable.
_probe_model, _probe_tok = load_policy(MODEL, trainable=True)
_trainable = [n for n, p in _probe_model.named_parameters() if p.requires_grad]
print(f"Loaded {MODEL.sft_adapter}. Trainable params: {len(_trainable)} tensors; sample:", _trainable[:3])
assert any("lora" in n.lower() for n in _trainable), "No LoRA params marked trainable — load path is wrong"
free_model(_probe_model)
del _probe_tok
gc.collect(); torch.cuda.empty_cache()
print("Model load path verified.")

## 10 · Baseline eval — SFT adapter, single-step env reward

Single-pass eval on the val set. This is the "before" column of the headline comparison. The richer *multi-step* eval happens later; this one is only here to confirm the SFT-loaded model outputs sane commands against the env **before** we start GRPO.

Written as a small `evaluate_single_step` helper because GRPO's inner trial loop needs the same logic.

In [ ]:
import json
import statistics as _stats
from dataclasses import asdict


@dataclass
class SingleStepMetrics:
    """One row of headline comparison numbers."""
    env_reward_mean:   float
    env_success_rate:  float   # fraction with reward >= 1.0
    format_pct:        float
    n:                 int

    def as_dict(self) -> dict:
        return asdict(self)


def evaluate_single_step(model, tokenizer, dataset, env: EnvRewardClient,
                         task_map: dict[int, Task],
                         max_new_tokens: int = 128) -> SingleStepMetrics:
    """Generate one command per prompt, score against the env, summarize."""
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    formats: list[float] = []
    tasks_to_score: list[Task] = []
    cmds_to_score:  list[str]  = []

    for row in dataset:
        prompt_text = tokenizer.apply_chat_template(
            row["prompt"], tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        formats.append(1.0 if text.lstrip().startswith("aws ") else 0.0)
        tasks_to_score.append(task_map[int(row["task_id"])])
        cmds_to_score.append(extract_aws_command(text))

    # Score all env rewards in parallel across the 8 server slots
    rewards = env.score_batch(tasks_to_score, cmds_to_score)

    FastLanguageModel.for_training(model)

    return SingleStepMetrics(
        env_reward_mean=float(_stats.mean(rewards)),
        env_success_rate=sum(r >= 1.0 for r in rewards) / len(rewards),
        format_pct=float(_stats.mean(formats)),
        n=len(rewards),
    )


# Run the SFT-only baseline and persist it alongside Optuna + checkpoints
_baseline_model, _baseline_tok = load_policy(MODEL, trainable=False)
baseline_metrics = evaluate_single_step(
    _baseline_model, _baseline_tok, VAL_DS, ENV_CLIENT, TASK_MAP,
    max_new_tokens=TRAIN.max_completion_length,
)
(OUT_DIR / "baseline_single_step.json").write_text(json.dumps(baseline_metrics.as_dict(), indent=2))
free_model(_baseline_model); del _baseline_tok
gc.collect(); torch.cuda.empty_cache()

print("SFT baseline (single-step, val):", baseline_metrics.as_dict())

## 11 · GRPOTrainer factory

`build_trainer` is a pure factory: given a typed `TrainingConfig`, it produces a ready `GRPOTrainer`. The trial objective and the final-run cell both call this — one code path for both.

Critical: `remove_unused_columns=False` preserves `task_id` through TRL's data collator so it reaches `env_reward` as a kwarg. This is the #1 GRPOTrainer gotcha.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback


class EnvHealthCallback(TrainerCallback):
    """Log env health counters + drain them every N steps.

    Also provides an early-warning bail-out: if every scoring call in a
    window came back as timeout/conn_err, the hosted env is probably down
    and we want to stop training rather than polluting the adapter with
    zero-reward updates.
    """

    def __init__(self, env_client: EnvRewardClient, probe_every: int = 50,
                 fail_threshold: int = 32) -> None:
        self.env = env_client
        self.probe_every = probe_every
        self.fail_threshold = fail_threshold

    def on_log(self, args, state, control, logs=None, **kw):
        if state.global_step == 0 or state.global_step % self.probe_every != 0:
            return
        counters = self.env.drain_counters()
        try:
            wandb.log({f"env/{k}": v for k, v in counters.items()}, step=state.global_step)
        except Exception:
            pass
        if counters["timeout"] + counters["conn_err"] >= self.fail_threshold:
            print(
                f"[EnvHealthCallback] {counters} at step {state.global_step}. "
                "Stopping training — check the hosted env / tunnel."
            )
            control.should_training_stop = True


def build_trainer(model, tokenizer, train_ds, eval_ds,
                  reward_funcs: Iterable[Callable],
                  cfg: TrainingConfig, *,
                  output_dir: str, wandb_run_name: str,
                  use_fp16: bool, use_bf16: bool,
                  max_steps: int | None = None,
                  save_strategy: str = "steps") -> GRPOTrainer:
    """Assemble a GRPOTrainer from a typed TrainingConfig."""
    args = GRPOConfig(
        output_dir=output_dir, run_name=wandb_run_name,
        num_generations=cfg.num_generations, beta=cfg.beta,
        temperature=cfg.temperature, top_p=cfg.top_p,
        max_prompt_length=cfg.max_prompt_length,
        max_completion_length=cfg.max_completion_length,
        learning_rate=cfg.learning_rate, lr_scheduler_type="cosine",
        optim="adamw_8bit", weight_decay=0.0, max_grad_norm=1.0,
        warmup_ratio=cfg.warmup_ratio,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        num_train_epochs=cfg.num_train_epochs,
        max_steps=(max_steps if max_steps is not None else -1),
        fp16=use_fp16, bf16=use_bf16,
        eval_strategy="steps", eval_steps=cfg.eval_steps,
        save_strategy=save_strategy, save_steps=cfg.save_steps,
        save_total_limit=cfg.save_total_limit,
        logging_steps=1, report_to="wandb", seed=cfg.seed,
        remove_unused_columns=False,   # CRITICAL: preserves task_id for reward_fns
    )
    return GRPOTrainer(
        model=model, processing_class=tokenizer,
        reward_funcs=list(reward_funcs),
        reward_weights=[1.0, 0.15, 0.05],
        args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        callbacks=[EnvHealthCallback(ENV_CLIENT)],
    )


print("GRPOTrainer factory ready.")

## 12 · Optuna hyperparameter search

Six short trials (30 GRPO steps each, 80-row training subset). Each trial returns the mean **env reward on the held-out val set** — the one metric we actually want to maximize.

- **Resumable**: sqlite storage + `load_if_exists=True`. A dropped Colab session picks up where it left off.
- **Pruned**: `MedianPruner` — kill trials that are trending below the median after 10 steps.
- **Search space** chosen to bracket the GRPO-on-3B defaults reported in the DeepSeek-Math paper.

In [ ]:
import optuna


def suggest_training_config(trial: optuna.Trial, defaults: TrainingConfig) -> TrainingConfig:
    """Return a mutated copy of `defaults` with Optuna-sampled hparams.

    Keeping all sampling in one place makes the search space auditable —
    one function, one diff, no magic kwargs buried in the trainer config.
    """
    return replace(
        defaults,
        learning_rate   = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        beta            = trial.suggest_float("beta", 0.0, 0.1),
        num_generations = trial.suggest_categorical("num_generations", [4, 8]),
        temperature     = trial.suggest_float("temperature", 0.7, 1.0),
        lora_r          = trial.suggest_categorical("lora_r", [8, 16, 32]),
        lora_alpha_mul  = trial.suggest_categorical("lora_alpha_mul", [1, 2]),
    )


def trial_objective(trial: optuna.Trial) -> float:
    """Short GRPO run + val eval. Returns mean env reward on VAL_DS."""
    trial_cfg = suggest_training_config(trial, TRAIN)
    output_dir = str(OUT_DIR / f"optuna/trial-{trial.number}")

    # Fresh model per trial so LoRA r/α can actually change between trials
    model, tokenizer = load_policy(MODEL, trainable=True)

    # Per-trial wandb run, cleanly finished at the end
    run = wandb.init(
        project=PIPE.wandb_project, entity=PIPE.wandb_entity,
        name=f"optuna-trial-{trial.number}",
        config={**asdict(trial_cfg), "trial_number": trial.number},
        reinit=True,
    )

    trainer = build_trainer(
        model, tokenizer,
        train_ds=TRAIN_DS.shuffle(seed=trial.number).select(
            range(min(PIPE.trial_train_subset, len(TRAIN_DS)))
        ),
        eval_ds=VAL_DS,
        reward_funcs=(ENV_REWARD, FORMAT_REWARD, LENGTH_REWARD),
        cfg=trial_cfg,
        output_dir=output_dir,
        wandb_run_name=run.name,
        use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
        max_steps=PIPE.trial_max_steps,
        save_strategy="no",   # no checkpointing during trials
    )

    try:
        trainer.train()
        metrics = evaluate_single_step(
            trainer.model, tokenizer, VAL_DS, ENV_CLIENT, TASK_MAP,
            max_new_tokens=trial_cfg.max_completion_length,
        )
        score = metrics.env_reward_mean
        wandb.log({"trial/env_reward_mean": score,
                   "trial/env_success_rate": metrics.env_success_rate})
    finally:
        wandb.finish()
        free_model(trainer); free_model(model); del tokenizer
        gc.collect(); torch.cuda.empty_cache()

    # Report + prune hook
    trial.report(score, step=PIPE.trial_max_steps)
    if trial.should_prune():
        raise optuna.TrialPruned()
    return score


print("Optuna objective defined.")

## 13 · Run the study + visualize

Plots saved into `OUT_DIR` and logged to the parent wandb run. If the notebook disconnects mid-study, re-running this cell resumes from the sqlite file.

In [ ]:
STUDY_DB = OUT_DIR / "optuna.db"
STUDY_NAME = "aws-rl-grpo-search"

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=TRAIN.seed),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=10),
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    load_if_exists=True,
)

completed = sum(1 for t in study.trials if t.state.name == "COMPLETE")
remaining = max(0, PIPE.n_trials - completed)
print(f"Optuna study '{STUDY_NAME}': {completed} completed, {remaining} remaining.")

if remaining > 0:
    study.optimize(trial_objective, n_trials=remaining)

best_cfg = replace(TRAIN, **{
    k: v for k, v in study.best_params.items() if k in TrainingConfig.__dataclass_fields__
})
(OUT_DIR / "optuna_best.json").write_text(json.dumps({
    "best_value": study.best_value,
    "best_params": study.best_params,
    "resolved_config": asdict(best_cfg),
}, indent=2))

print(f"Best trial env_reward_mean = {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
import plotly.io as pio

try:
    fig_history  = optuna.visualization.plot_optimization_history(study)
    fig_parallel = optuna.visualization.plot_parallel_coordinate(study)
    fig_import   = optuna.visualization.plot_param_importances(study)
    for name, fig in (("history", fig_history), ("parallel", fig_parallel), ("importances", fig_import)):
        out_png = OUT_DIR / f"optuna_{name}.png"
        pio.write_image(fig, str(out_png), width=900, height=500)
        print(f"Saved {out_png}")
    # Show inline as well
    fig_history.show(); fig_parallel.show(); fig_import.show()
except Exception as e:
    print(f"Optuna plots skipped: {e}")

## 14 · Final GRPO run — best hparams, full data, checkpointed

Uses the Optuna winner on the full 1-epoch training set with periodic checkpoints (`save_steps=25`, `save_total_limit=3`). `CheckpointManager.latest()` auto-detects a prior checkpoint so re-running this cell after a disconnect resumes safely.

Wandb run `final-grpo` gets a stable run id so resumed training stitches into the same panel instead of creating a fresh chart.

In [ ]:
class CheckpointManager:
    """Find the latest `checkpoint-N/` under `root` for safe resume."""

    def __init__(self, root: Path) -> None:
        self.root = root

    def latest(self) -> str | None:
        if not self.root.exists():
            return None
        ckpts = sorted(
            (d for d in self.root.glob("checkpoint-*") if d.is_dir()),
            key=lambda d: int(d.name.split("-")[-1]),
        )
        return str(ckpts[-1]) if ckpts else None


FINAL_RUN_DIR = OUT_DIR / "final_grpo"
ckpt_mgr = CheckpointManager(FINAL_RUN_DIR)
resume_from = ckpt_mgr.latest()

# Stable wandb run id across resumes — same dashboard panel
(run_id_file := OUT_DIR / "wandb_final_run_id.txt")
if run_id_file.exists():
    os.environ["WANDB_RUN_ID"] = run_id_file.read_text().strip()
    os.environ["WANDB_RESUME"] = "allow"
else:
    import uuid
    new_id = uuid.uuid4().hex[:8]
    run_id_file.write_text(new_id)
    os.environ["WANDB_RUN_ID"] = new_id

final_model, final_tok = load_policy(MODEL, trainable=True)
final_run = wandb.init(
    project=PIPE.wandb_project, entity=PIPE.wandb_entity,
    name="final-grpo",
    id=os.environ["WANDB_RUN_ID"],
    resume="allow",
    config={**asdict(best_cfg), "phase": "final", "resume_from": resume_from},
)

final_trainer = build_trainer(
    final_model, final_tok,
    train_ds=TRAIN_DS,
    eval_ds=VAL_DS,
    reward_funcs=(ENV_REWARD, FORMAT_REWARD, LENGTH_REWARD),
    cfg=best_cfg,
    output_dir=str(FINAL_RUN_DIR),
    wandb_run_name=final_run.name,
    use_fp16=RT.use_fp16, use_bf16=RT.use_bf16,
    save_strategy="steps",
)

if resume_from:
    print(f"Resuming GRPO from {resume_from}")
final_trainer.train(resume_from_checkpoint=resume_from)

# Persist the final LoRA adapter locally before anything else touches VRAM
adapter_local = OUT_DIR / "grpo_adapter"
final_trainer.model.save_pretrained(str(adapter_local))
final_tok.save_pretrained(str(adapter_local))
print(f"Saved GRPO adapter to {adapter_local}")

## 15 · Multi-step evaluation harness

Training was single-step for TRL compatibility; *evaluation* runs full episodes so we can measure:

- per-tier task success
- hints used per solved task
- recovery rate after chaos injection
- drift repair rate
- steps to solve
- rollback count (safety)
- generalization gap (reserve vs train-held-out)

The pattern mirrors [`scripts/grpo_train.py`](https://github.com/UdayKiranPadhy/aws-rl-env/blob/master/scripts/grpo_train.py)'s `run_single_rollout`, but uses `GrpoPool` for 8-way concurrent rollouts so a 100-task eval finishes in ~7 minutes.

In [ ]:
import asyncio
from collections import defaultdict
from models import AwsRlAction
from client import AwsRlEnv
from scripts.grpo_pool import GrpoPool


SYSTEM_PROMPT = (
    "You are an expert AWS SRE agent. You operate a simulated AWS cloud by "
    "emitting one AWS CLI command at a time. You will see the task description "
    "and the most recent command output, then reply with EXACTLY ONE AWS CLI "
    "command on a single line starting with 'aws '. No explanation, no markdown, "
    "no quotes \u2014 just the command."
)


def build_multi_step_prompt(tokenizer, task: Task,
                             history: list[tuple[str, str]]) -> str:
    """Assemble chat-style prompt including the last few (cmd, output) turns."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"TASK: {task.description}"},
    ]
    for cmd, out in history[-4:]:   # last 4 turns fit in 512 tokens
        messages.append({"role": "assistant", "content": cmd})
        messages.append({"role": "user",      "content": f"OUTPUT:\n{out[:400]}"})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


@dataclass
class EpisodeResult:
    task_id: int
    tier: str
    is_drift: bool
    achieved: bool
    terminal_reward: float
    steps_taken: int
    hints_used: int
    chaos_occurred: bool
    command_failures: int


async def run_episode(env: AwsRlEnv, model, tokenizer,
                      task: Task, drift_ids: set[int],
                      max_steps: int = 15,
                      max_new_tokens: int = 96) -> EpisodeResult:
    """Roll one episode against one env session. Sampling temperature is low
    to reflect deployment behaviour rather than training-time exploration."""
    device = next(model.parameters()).device
    res = await env.reset(task=task)
    history: list[tuple[str, str]] = []
    steps_taken = 0
    command_failures = 0
    terminal_reward = 0.0
    achieved = False

    for _ in range(max_steps):
        steps_taken += 1
        prompt = build_multi_step_prompt(tokenizer, task, history)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.4,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(
            ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
        )
        cmd = extract_aws_command(text)
        res = await env.step(AwsRlAction(command=cmd))
        terminal_reward = float(res.reward)
        obs = res.observation
        if not obs.command_success:
            command_failures += 1
        history.append((cmd, obs.command_output or ""))
        if obs.task_achieved:
            achieved = True
        if res.done:
            break

    # One final /state poll for chaos flag — TrackerState doesn't expose
    # rollback counts, so we skip that metric rather than report zeros.
    try:
        state = await env.state()
        chaos = bool(getattr(state, "chaos_occurred", False))
    except Exception:
        chaos = False

    return EpisodeResult(
        task_id=int(task.task_id),
        tier=task.difficulty.value,
        is_drift=int(task.task_id) in drift_ids,
        achieved=achieved,
        terminal_reward=terminal_reward,
        steps_taken=steps_taken,
        hints_used=int(getattr(res.observation, "hints_used", 0) or 0),
        chaos_occurred=chaos,
        command_failures=command_failures,
    )


print("run_episode defined.")

In [ ]:
@dataclass
class RichMetrics:
    """All the metrics the hackathon judges care about."""
    success_by_tier:       dict
    reward_by_tier:        dict
    overall_success_rate:  float
    overall_reward_mean:   float
    hints_per_solved:      float
    recovery_rate:         float
    drift_repair_rate:     float
    steps_to_solve:        float
    destructive_fail_rate: float
    n_episodes:            int

    def as_dict(self) -> dict:
        return asdict(self)


def summarize_episodes(results: list[EpisodeResult]) -> RichMetrics:
    """Aggregate per-tier and overall stats from a list of EpisodeResults.

    Drift detection uses the per-result `is_drift` flag (set from DRIFT_TASK_IDS)
    rather than a tier string — drift tasks live inside the EXPERT tier files.
    """
    by_tier: dict[str, list[EpisodeResult]] = defaultdict(list)
    for r in results:
        by_tier[r.tier].append(r)

    success_by_tier = {tier: sum(r.achieved for r in xs) / max(1, len(xs))
                       for tier, xs in by_tier.items()}
    reward_by_tier  = {tier: (sum(r.terminal_reward for r in xs) / max(1, len(xs)))
                       for tier, xs in by_tier.items()}

    solved = [r for r in results if r.achieved]
    chaos_episodes = [r for r in results if r.chaos_occurred]
    drift_episodes = [r for r in results if r.is_drift]

    return RichMetrics(
        success_by_tier=success_by_tier,
        reward_by_tier=reward_by_tier,
        overall_success_rate=len(solved) / max(1, len(results)),
        overall_reward_mean=sum(r.terminal_reward for r in results) / max(1, len(results)),
        hints_per_solved=(sum(r.hints_used for r in solved) / len(solved)) if solved else 0.0,
        recovery_rate=(sum(r.achieved for r in chaos_episodes) / len(chaos_episodes))
                      if chaos_episodes else 0.0,
        drift_repair_rate=(sum(r.achieved for r in drift_episodes) / len(drift_episodes))
                          if drift_episodes else 0.0,
        steps_to_solve=(sum(r.steps_taken for r in solved) / len(solved)) if solved else 0.0,
        destructive_fail_rate=sum(r.command_failures > 0 for r in results) / max(1, len(results)),
        n_episodes=len(results),
    )


async def evaluate_multi_step(base_url: str, task_ids: list[int],
                              task_map: dict[int, Task],
                              drift_ids: set[int],
                              model, tokenizer,
                              pool_size: int = 8,
                              max_steps: int = 15) -> RichMetrics:
    """Run one episode per task_id across `pool_size` concurrent env sessions."""
    results: list[EpisodeResult] = []
    async with GrpoPool(base_url=base_url, size=pool_size) as pool:
        for start in range(0, len(task_ids), pool_size):
            chunk = task_ids[start:start + pool_size]
            coros = [run_episode(env, model, tokenizer, task_map[tid], drift_ids,
                                  max_steps=max_steps)
                     for env, tid in zip(pool.envs, chunk)]
            chunk_results = await asyncio.gather(*coros, return_exceptions=True)
            for r in chunk_results:
                if isinstance(r, EpisodeResult):
                    results.append(r)
                else:
                    print(f"[eval] episode raised {type(r).__name__}: {r}")
    return summarize_episodes(results)


def select_eval_task_ids(reserve_ds, drift_ids: set[int], cap: int) -> list[int]:
    """Task ids for the multi-step eval: reserve split + every drift task."""
    tids = [int(r["task_id"]) for r in reserve_ds][:cap] if reserve_ds else []
    for did in drift_ids:
        if did not in tids:
            tids.append(did)
    return tids


print("evaluate_multi_step defined.")

## 16 · Before / after multi-step evaluation

Runs the rich multi-step evaluator once with the **SFT-only** model (baseline) and once with the **final GRPO** model, then prints a delta table and logs every metric to wandb under the `eval/*` and `eval/delta_*` namespaces for judge-facing charts.

In [ ]:
def _flatten_metrics(m: RichMetrics, prefix: str) -> dict:
    out = {}
    for k, v in m.as_dict().items():
        if isinstance(v, dict):
            for tier, val in v.items():
                out[f"{prefix}/{k}/{tier}"] = val
        else:
            out[f"{prefix}/{k}"] = v
    return out


def _delta_metrics(before: RichMetrics, after: RichMetrics) -> dict:
    b, a = before.as_dict(), after.as_dict()
    delta: dict[str, float] = {}
    for k in a:
        if isinstance(a[k], dict):
            for tier, v in a[k].items():
                bv = b.get(k, {}).get(tier, 0.0)
                delta[f"eval/delta_{k}/{tier}"] = v - bv
        elif isinstance(a[k], (int, float)):
            delta[f"eval/delta_{k}"] = a[k] - b[k]
    return delta


EVAL_TASK_IDS = select_eval_task_ids(RESERVE_DS, DRIFT_TASK_IDS, cap=PIPE.eval_reserve_cap)
print(f"Multi-step eval on {len(EVAL_TASK_IDS)} tasks "
      f"({sum(1 for t in EVAL_TASK_IDS if t in DRIFT_TASK_IDS)} drift).")

# --- SFT baseline ---
print("Evaluating SFT baseline (multi-step)...")
sft_model, sft_tok = load_policy(MODEL, trainable=False)
sft_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    sft_model, sft_tok, pool_size=PIPE.env_pool_size,
)
free_model(sft_model); del sft_tok
gc.collect(); torch.cuda.empty_cache()
(OUT_DIR / "baseline_multi_step.json").write_text(json.dumps(sft_metrics.as_dict(), indent=2))

# --- GRPO after ---
print("Evaluating GRPO-trained model (multi-step)...")
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(final_trainer.model)
grpo_metrics = await evaluate_multi_step(
    ENV_BASE_URL, EVAL_TASK_IDS, TASK_MAP, DRIFT_TASK_IDS,
    final_trainer.model, final_tok, pool_size=PIPE.env_pool_size,
)
(OUT_DIR / "grpo_multi_step.json").write_text(json.dumps(grpo_metrics.as_dict(), indent=2))

# --- Deltas + wandb ---
deltas = _delta_metrics(sft_metrics, grpo_metrics)
wandb.log({
    **_flatten_metrics(sft_metrics,  "eval/sft"),
    **_flatten_metrics(grpo_metrics, "eval/grpo"),
    **deltas,
})

# Judge-facing table
def _render_row(name, b, a):
    return f"| {name:<26} | {b:>12.3f} | {a:>12.3f} | {a - b:+.3f} |"

print("\n| Metric                     | SFT baseline | GRPO        | Delta  |")
print("|----------------------------|-------------:|------------:|-------:|")
print(_render_row("overall_success_rate",  sft_metrics.overall_success_rate,  grpo_metrics.overall_success_rate))
print(_render_row("overall_reward_mean",   sft_metrics.overall_reward_mean,   grpo_metrics.overall_reward_mean))
print(_render_row("hints_per_solved",      sft_metrics.hints_per_solved,      grpo_metrics.hints_per_solved))
print(_render_row("recovery_rate",         sft_metrics.recovery_rate,         grpo_metrics.recovery_rate))
print(_render_row("drift_repair_rate",     sft_metrics.drift_repair_rate,     grpo_metrics.drift_repair_rate))
print(_render_row("steps_to_solve",        sft_metrics.steps_to_solve,        grpo_metrics.steps_to_solve))
print(_render_row("destructive_fail_rate", sft_metrics.destructive_fail_rate, grpo_metrics.destructive_fail_rate))

for tier in sorted(set(sft_metrics.success_by_tier) | set(grpo_metrics.success_by_tier)):
    b = sft_metrics.success_by_tier.get(tier, 0.0)
    a = grpo_metrics.success_by_tier.get(tier, 0.0)
    print(_render_row(f"success[{tier}]", b, a))

## 17 · Qualitative rollouts — one task per tier

A small set of showable transcripts (task, generated command, env output, reward). Judges love seeing actual agent behaviour, not just numbers.

In [ ]:
async def qualitative_rollouts(model, tokenizer, task_map: dict[int, Task],
                                drift_ids: set[int],
                                samples_per_tier: int = 1) -> list[dict]:
    """Pick one task per difficulty tier and print a full rollout transcript."""
    per_tier: dict[str, list[Task]] = defaultdict(list)
    for t in task_map.values():
        per_tier[t.difficulty.value].append(t)
    chosen: list[Task] = []
    for tier in ["warmup", "beginner", "intermediate", "advanced", "expert"]:
        if per_tier.get(tier):
            chosen.extend(per_tier[tier][:samples_per_tier])

    transcripts = []
    async with GrpoPool(base_url=ENV_BASE_URL, size=min(len(chosen), PIPE.env_pool_size)) as pool:
        for env, task in zip(pool.envs, chosen):
            ep = await run_episode(env, model, tokenizer, task, drift_ids, max_steps=8)
            transcripts.append({
                "tier": task.difficulty.value,
                "task_id": int(task.task_id),
                "description": task.description,
                "achieved": ep.achieved,
                "terminal_reward": ep.terminal_reward,
                "steps_taken": ep.steps_taken,
            })
    return transcripts


qual = await qualitative_rollouts(final_trainer.model, final_tok, TASK_MAP, DRIFT_TASK_IDS)
print(json.dumps(qual, indent=2, default=str))
(OUT_DIR / "qualitative_rollouts.json").write_text(json.dumps(qual, indent=2, default=str))

## 18 · Publish the GRPO adapter to a new HF Hub repo

Pushes **only** to `Sizzing/aws-rl-grpo-qwen25coder3b-adapter`. The existing SFT adapter repo `Sizzing/aws-rl-sft-qwen25coder3b-adapter` is never touched — both coexist on Hub so reviewers can load either and compare side by side.

The model card notes the lineage (`base_model = SFT adapter`) so anyone opening the repo on Hub sees immediately that this is a second-stage RL fine-tune.

In [ ]:
from datetime import datetime, timezone


def write_model_card(adapter_dir: Path, model_spec: ModelSpec,
                     cfg: TrainingConfig, grpo: RichMetrics, sft: RichMetrics) -> Path:
    """Emit a README.md for the pushed repo documenting training recipe + metrics."""
    card = f"""---
library_name: peft
base_model: {model_spec.sft_adapter}
pipeline_tag: text-generation
tags: [grpo, lora, aws, reinforcement-learning]
---

# aws-rl-grpo-qwen25coder3b-adapter

GRPO (Group Relative Policy Optimization) LoRA adapter continuing from
[`{model_spec.sft_adapter}`](https://huggingface.co/{model_spec.sft_adapter}).
Trained single-step with live AWS RL env rewards; evaluated multi-step.

## How to load

```python
from unsloth import FastLanguageModel
from peft import PeftModel

base, tok = FastLanguageModel.from_pretrained(
    "{model_spec.base_model}", max_seq_length={model_spec.max_seq_length}, load_in_4bit=True,
)
model = PeftModel.from_pretrained(base, "{model_spec.grpo_adapter}")
FastLanguageModel.for_inference(model)
```

## Training recipe

| Knob                  | Value              |
|-----------------------|--------------------|
| learning_rate         | {cfg.learning_rate:.2e}   |
| beta (KL coef)        | {cfg.beta:.3f}             |
| num_generations (G)   | {cfg.num_generations}                  |
| temperature           | {cfg.temperature:.2f}              |
| lora_r / alpha_mul    | {cfg.lora_r} / {cfg.lora_alpha_mul}                   |
| max_completion_length | {cfg.max_completion_length}                |
| per-device batch      | {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} accum       |

## Multi-step eval (reserve + drift)

| Metric                | SFT baseline | GRPO   | Delta   |
|-----------------------|-------------:|-------:|--------:|
| overall_success_rate  | {sft.overall_success_rate:.3f}        | {grpo.overall_success_rate:.3f}  | {grpo.overall_success_rate - sft.overall_success_rate:+.3f} |
| overall_reward_mean   | {sft.overall_reward_mean:.3f}        | {grpo.overall_reward_mean:.3f}  | {grpo.overall_reward_mean - sft.overall_reward_mean:+.3f} |
| hints_per_solved      | {sft.hints_per_solved:.3f}        | {grpo.hints_per_solved:.3f}  | {grpo.hints_per_solved - sft.hints_per_solved:+.3f} |
| recovery_rate         | {sft.recovery_rate:.3f}        | {grpo.recovery_rate:.3f}  | {grpo.recovery_rate - sft.recovery_rate:+.3f} |
| drift_repair_rate     | {sft.drift_repair_rate:.3f}        | {grpo.drift_repair_rate:.3f}  | {grpo.drift_repair_rate - sft.drift_repair_rate:+.3f} |
| steps_to_solve        | {sft.steps_to_solve:.3f}        | {grpo.steps_to_solve:.3f}  | {grpo.steps_to_solve - sft.steps_to_solve:+.3f} |

Trained {datetime.now(timezone.utc).isoformat(timespec='minutes')} on {RT.gpu_name}.
"""
    card_path = adapter_dir / "README.md"
    card_path.write_text(card)
    return card_path


# Write card locally, then push both adapter files + card
adapter_local = OUT_DIR / "grpo_adapter"
write_model_card(adapter_local, MODEL, best_cfg, grpo_metrics, sft_metrics)

commit_msg = f"GRPO run {datetime.now(timezone.utc).isoformat(timespec='minutes')}"
final_trainer.model.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
final_tok.push_to_hub(
    MODEL.grpo_adapter, private=True, token=HF_TOKEN, commit_message=commit_msg,
)
HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=str(adapter_local / "README.md"),
    path_in_repo="README.md",
    repo_id=MODEL.grpo_adapter,
    commit_message=commit_msg,
)
print(f"Pushed to https://huggingface.co/{MODEL.grpo_adapter}")
print(f"SFT adapter at {MODEL.sft_adapter} untouched — both models available on Hub.")

## 19 · Clean shutdown + artifact bundle

Closes wandb, releases the GPU, tars `OUT_DIR` into a single downloadable archive (Colab `files.download`). Nothing else needs to be killed — the env server is hosted externally.

In [ ]:
import tarfile


def bundle_artifacts(out_dir: Path) -> Path:
    """Tar up the run's artifacts for easy download off the VM."""
    archive = out_dir.parent / f"{out_dir.name}.tar.gz"
    keep = ("grpo_adapter", "optuna.db", "optuna_best.json",
            "baseline_single_step.json", "baseline_multi_step.json",
            "grpo_multi_step.json", "qualitative_rollouts.json",
            "wandb_final_run_id.txt", "optuna_history.png",
            "optuna_parallel.png", "optuna_importances.png")
    with tarfile.open(archive, "w:gz") as tar:
        for name in keep:
            p = out_dir / name
            if p.exists():
                tar.add(p, arcname=name)
    return archive


# Release GPU + wandb first
free_model(final_trainer)
del final_trainer, final_model, final_tok
gc.collect(); torch.cuda.empty_cache()

try:
    wandb.finish()
except Exception as e:
    print(f"wandb.finish warning: {e}")

archive = bundle_artifacts(OUT_DIR)
print(f"\nBundle: {archive}")
print(f"\nHF Hub:\n  SFT:  https://huggingface.co/{MODEL.sft_adapter}\n  GRPO: https://huggingface.co/{MODEL.grpo_adapter}")

if IS_COLAB:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as e:
        print(f"Colab auto-download skipped: {e}\nDownload manually from {archive}")